# Origami RL — GRPO Training Notebook

Train an LLM to generate valid FOLD-format crease patterns that fold into target shapes.

**Pipeline:**
1. LLM receives a prompt describing a target shape (e.g. "fold diagonally into a triangle")
2. LLM generates a FOLD JSON crease pattern
3. Physics simulator folds the paper analytically
4. Reward = shape similarity (chamfer distance) to target × 20

**Reward functions:**
- `valid_fold`: +1.0 valid FOLD JSON, −0.5 parseable but invalid, −2.0 unparseable
- `shape_match`: similarity × 20.0 (0–20), −1.0 sim fails, −2.0 invalid FOLD

**Algorithm:** GRPO (Group Relative Policy Optimization) via TRL + Unsloth LoRA

## 1. Install Dependencies

In [ ]:
# Run this cell once to install all dependencies
# For Colab: unsloth has a specific install process
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Unsloth's recommended Colab install
    !pip install --no-deps "unsloth[colab-new]"
    !pip install --no-deps trl datasets peft accelerate bitsandbytes xformers
else:
    !pip install -q "trl>=0.7" "datasets>=2.14" unsloth torch transformers accelerate bitsandbytes

# Core origami env deps (numpy, scipy, pydantic)
!pip install -q numpy scipy pydantic

## 2. Setup Python Path & Imports

In [ ]:
import os
import sys
import json

# Add the repo root to Python path so origami_server and training modules are importable
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Repo root: {REPO_ROOT}")
print(f"Python: {sys.version}")

In [ ]:
import numpy as np

# Verify origami env modules load correctly
from origami_server.tasks import TASKS, get_task, list_tasks
from origami_server.engine.fold_parser import validate_fold, parse_fold
from origami_server.engine.simulate import simulate
from origami_server.engine.shape_match import compute_shape_match
from training.reward import valid_fold, shape_match, extract_fold_json

print(f"Available tasks: {list_tasks()}")
print("All origami modules loaded successfully.")

## 3. Explore the Environment

Sanity-check the simulator and reward functions before training.

In [ ]:
# Print all tasks with their details
for name, task in TASKS.items():
    print(f"\n{'='*50}")
    print(f"Task: {task['name']}")
    print(f"Description: {task['description']}")
    print(f"Difficulty: {task['difficulty']}")
    print(f"Paper: {task['paper']}")
    fold = task["target_fold"]
    n_verts = len(fold["vertices_coords"])
    n_edges = len(fold["edges_vertices"])
    n_folds = sum(1 for a in fold["edges_assignment"] if a in ("M", "V"))
    print(f"Vertices: {n_verts}, Edges: {n_edges}, Fold creases: {n_folds}")

In [ ]:
# Test the simulator on each task
for name in list_tasks():
    task = get_task(name)
    target_fold = task["target_fold"]
    
    # Simulate flat (0%), half (50%), and fully folded (100%)
    r_flat = simulate(target_fold, crease_percent=0.0)
    r_half = simulate(target_fold, crease_percent=0.5)
    r_full = simulate(target_fold, crease_percent=1.0)
    
    z_half = r_half.positions[:, 2].max() - r_half.positions[:, 2].min()
    
    # Shape match: target vs itself should be 1.0
    self_sim = compute_shape_match(r_full.positions, r_full.positions)
    
    print(f"{name:15s} | converged={r_full.converged} | strain={r_full.max_strain:.6f} | "
          f"z_range@50%={z_half:.3f} | self_similarity={self_sim:.3f}")

In [ ]:
# Test reward functions with mock LLM outputs
triangle_fold = TASKS["triangle"]["target_fold"]

# Simulate what the reward functions see during training:
# completions = list of [{"content": "...LLM response..."}]
good_response = json.dumps(triangle_fold)
bad_json = "I think we should fold it like this..."
invalid_fold = json.dumps({"vertices_coords": [[0, 0]], "edges_vertices": [], "edges_assignment": []})

completions = [
    [{"content": f"```json\n{good_response}\n```"}],   # correct answer in fenced block
    [{"content": bad_json}],                              # garbage
    [{"content": invalid_fold}],                          # parseable but invalid FOLD
]

print("valid_fold rewards:", valid_fold(completions))
print("shape_match rewards:", shape_match(completions, task_name="triangle"))
print()
print("Expected: valid_fold  = [1.0, -2.0, -0.5]")
print("Expected: shape_match = [20.0, -2.0, -1.0]")

## 4. Visualize Tasks

2D crease patterns for each task (matplotlib).

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

EDGE_COLORS = {"M": "red", "V": "blue", "B": "black"}
EDGE_STYLES = {"M": "--", "V": ":", "B": "-"}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, (name, task) in enumerate(TASKS.items()):
    fold = task["target_fold"]
    verts = np.array(fold["vertices_coords"])
    
    # Row 1: 2D crease pattern
    ax = axes[0, idx]
    ax.set_title(f"{name}\n{task['description']}", fontsize=9)
    ax.set_aspect("equal")
    ax.set_xlim(-0.1, 1.1)
    ax.set_ylim(-0.1, 1.1)
    ax.grid(True, alpha=0.2)
    
    for i, (e, a) in enumerate(zip(fold["edges_vertices"], fold["edges_assignment"])):
        v1, v2 = verts[e[0]], verts[e[1]]
        color = EDGE_COLORS.get(a, "gray")
        style = EDGE_STYLES.get(a, "-")
        lw = 2.5 if a == "B" else 1.8
        ax.plot([v1[0], v2[0]], [v1[1], v2[1]], color=color, linestyle=style, linewidth=lw)
    
    ax.scatter(verts[:, 0], verts[:, 1], c="black", s=15, zorder=5)
    
    # Row 2: 3D folded shape
    ax3 = fig.add_subplot(2, 4, idx + 5, projection="3d")
    result = simulate(fold, crease_percent=1.0)
    pos = result.positions
    
    if "faces_vertices" in fold:
        for face in fold["faces_vertices"]:
            tri_verts = [pos[vi] for vi in face]
            poly = Poly3DCollection([tri_verts], alpha=0.3, facecolor="lightskyblue", edgecolor="steelblue")
            ax3.add_collection3d(poly)
    
    for i, (e, a) in enumerate(zip(fold["edges_vertices"], fold["edges_assignment"])):
        p1, p2 = pos[e[0]], pos[e[1]]
        color = EDGE_COLORS.get(a, "gray")
        ax3.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color=color, linewidth=1.2)
    
    ax3.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c="black", s=10, zorder=5)
    ax3.set_title(f"Folded (3D)", fontsize=9)
    ax3.set_xlim(-0.2, 1.2)
    ax3.set_ylim(-0.2, 1.2)
    ax3.set_zlim(-0.6, 0.6)
    
    # Remove the empty 2D subplot that was in row 2
    axes[1, idx].remove()

plt.tight_layout()
plt.show()

## 5. Training Configuration

In [ ]:
# ============================================================
# Training hyperparameters — edit these before launching
# ============================================================

TASK_NAME = "triangle"          # "triangle", "half_fold", "quarter_fold", "letter_fold"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Change to your preferred model
MAX_STEPS = 600                 # Total GRPO training steps
NUM_GENERATIONS = 4             # Completions per prompt per step
LEARNING_RATE = 2e-4
LORA_R = 8                     # LoRA rank
LORA_ALPHA = 16                # LoRA alpha
MAX_PROMPT_LENGTH = 1024
MAX_COMPLETION_LENGTH = 1024
DATASET_SIZE = 1000             # Number of prompt copies (same prompt repeated)
OUTPUT_DIR = "outputs"
SAVE_STEPS = 100

## 6. Build the Prompt & Dataset

In [ ]:
from training.train_grpo import PROMPT_TEMPLATE, build_prompt

task = get_task(TASK_NAME)
prompt_text = build_prompt(task)

print("="*60)
print("PROMPT THAT THE LLM WILL SEE:")
print("="*60)
print(prompt_text)

In [ ]:
from datasets import Dataset

# GRPO pattern: same prompt repeated many times, the RL loop generates
# multiple completions per prompt and uses relative rewards to update policy
dataset = Dataset.from_list(
    [
        {
            "prompt": [{"role": "user", "content": prompt_text}],
            "answer": 0,  # placeholder, not used by GRPO
        }
    ]
    * DATASET_SIZE
)

print(f"Dataset size: {len(dataset)}")
print(f"Sample prompt (first 100 chars): {dataset[0]['prompt'][0]['content'][:100]}...")

## 7. Load Model + LoRA

Uses Unsloth for fast 4-bit LoRA fine-tuning. Falls back to standard HuggingFace if Unsloth isn't available.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Apple MPS (Metal) available — note: Unsloth requires CUDA, will use HF fallback")
else:
    print("No GPU detected — training will be very slow")

In [ ]:
USE_UNSLOTH = False

try:
    from unsloth import FastLanguageModel
    USE_UNSLOTH = True
    print("Using Unsloth for fast LoRA loading")
except ImportError:
    print("Unsloth not available, using standard HuggingFace + PEFT")

if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        load_in_4bit=True,
        max_seq_length=MAX_PROMPT_LENGTH + MAX_COMPLETION_LENGTH,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha=LORA_ALPHA,
        use_gradient_checkpointing="unsloth",
    )
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    ) if torch.cuda.is_available() else None

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto" if torch.cuda.is_available() else "cpu",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.print_trainable_parameters()

## 8. Setup GRPO Trainer

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Wrap shape_match to inject the task name
def shape_match_reward(completions, **kwargs):
    return shape_match(completions, task_name=TASK_NAME, **kwargs)

training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=MAX_STEPS,
    save_steps=SAVE_STEPS,
    output_dir=OUTPUT_DIR,
    report_to="none",  # Set to "wandb" if you want W&B logging
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[valid_fold, shape_match_reward],
    args=training_args,
    train_dataset=dataset,
)

print(f"Trainer ready. Task: {TASK_NAME}, Model: {MODEL_NAME}")
print(f"Max steps: {MAX_STEPS}, Generations per step: {NUM_GENERATIONS}")
print(f"Reward functions: valid_fold + shape_match")

## 9. Train!

In [ ]:
trainer.train()

## 10. Save the Trained Model

In [ ]:
SAVE_PATH = f"origami-{TASK_NAME}-lora"

# Save LoRA adapter
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to {SAVE_PATH}/")

# Optional: merge LoRA into base model and save full model
# merged_path = f"origami-{TASK_NAME}-merged"
# if USE_UNSLOTH:
#     model.save_pretrained_merged(merged_path, tokenizer)
# else:
#     merged_model = model.merge_and_unload()
#     merged_model.save_pretrained(merged_path)
#     tokenizer.save_pretrained(merged_path)
# print(f"Merged model saved to {merged_path}/")

## 11. Evaluate — Generate & Score Completions

Test the trained model by generating crease patterns and scoring them.

In [ ]:
# Put model in inference mode
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)

NUM_EVAL_SAMPLES = 8

# Build chat messages
messages = [{"role": "user", "content": prompt_text}]
input_ids = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to(model.device)

print(f"Generating {NUM_EVAL_SAMPLES} completions...")
print(f"Input length: {input_ids.shape[1]} tokens\n")

eval_completions = []
for i in range(NUM_EVAL_SAMPLES):
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    eval_completions.append([{"content": response}])
    
    # Quick score
    fold_data = extract_fold_json(response)
    if fold_data is None:
        status = "UNPARSEABLE"
    else:
        is_valid, err = validate_fold(fold_data)
        if not is_valid:
            status = f"INVALID: {err}"
        else:
            try:
                result = simulate(fold_data, crease_percent=1.0)
                target_result = simulate(task["target_fold"], crease_percent=1.0)
                sim = compute_shape_match(result.positions, target_result.positions)
                status = f"similarity={sim:.3f} (reward={sim * 20:.1f})"
            except Exception as e:
                status = f"SIM ERROR: {e}"
    
    print(f"  Sample {i+1}: {status}")

# Compute aggregate reward scores
print(f"\nAggregate rewards:")
vf_scores = valid_fold(eval_completions)
sm_scores = shape_match(eval_completions, task_name=TASK_NAME)
print(f"  valid_fold:  mean={np.mean(vf_scores):.2f}, scores={vf_scores}")
print(f"  shape_match: mean={np.mean(sm_scores):.2f}, scores={sm_scores}")

## 12. Visualize a Generated Fold

Pick the best completion and visualize its crease pattern + 3D fold vs the target.

In [ ]:
# Find the best valid completion
best_idx = int(np.argmax(sm_scores))
best_response = eval_completions[best_idx][0]["content"]
best_fold = extract_fold_json(best_response)

if best_fold is None or sm_scores[best_idx] <= 0:
    print("No valid completions to visualize.")
else:
    is_valid, _ = validate_fold(best_fold)
    if not is_valid:
        print("Best completion has invalid FOLD structure.")
    else:
        pred_result = simulate(best_fold, crease_percent=1.0)
        target_result = simulate(task["target_fold"], crease_percent=1.0)
        
        fig = plt.figure(figsize=(14, 5))
        
        # 1) Generated 2D crease pattern
        ax1 = fig.add_subplot(131)
        ax1.set_title(f"Generated Crease Pattern\n(sample {best_idx+1})", fontsize=10)
        ax1.set_aspect("equal")
        verts = np.array(best_fold["vertices_coords"])
        for i, (e, a) in enumerate(zip(best_fold["edges_vertices"], best_fold["edges_assignment"])):
            v1, v2 = verts[e[0]], verts[e[1]]
            color = EDGE_COLORS.get(a, "gray")
            style = EDGE_STYLES.get(a, "-")
            ax1.plot([v1[0], v2[0]], [v1[1], v2[1]], color=color, linestyle=style, linewidth=2)
        ax1.scatter(verts[:, 0], verts[:, 1], c="black", s=20, zorder=5)
        ax1.grid(True, alpha=0.2)
        
        # 2) Generated 3D fold
        ax2 = fig.add_subplot(132, projection="3d")
        ax2.set_title(f"Generated 3D Fold\nsimilarity={sm_scores[best_idx]/20:.3f}", fontsize=10)
        pos = pred_result.positions
        for i, (e, a) in enumerate(zip(best_fold["edges_vertices"], best_fold["edges_assignment"])):
            p1, p2 = pos[e[0]], pos[e[1]]
            color = EDGE_COLORS.get(a, "gray")
            ax2.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color=color, linewidth=1.5)
        ax2.scatter(pos[:, 0], pos[:, 1], pos[:, 2], c="black", s=15, zorder=5)
        
        # 3) Target 3D fold
        ax3 = fig.add_subplot(133, projection="3d")
        ax3.set_title("Target 3D Fold", fontsize=10)
        tpos = target_result.positions
        tfold = task["target_fold"]
        for i, (e, a) in enumerate(zip(tfold["edges_vertices"], tfold["edges_assignment"])):
            p1, p2 = tpos[e[0]], tpos[e[1]]
            color = EDGE_COLORS.get(a, "gray")
            ax3.plot([p1[0], p2[0]], [p1[1], p2[1]], [p1[2], p2[2]], color=color, linewidth=1.5)
        ax3.scatter(tpos[:, 0], tpos[:, 1], tpos[:, 2], c="black", s=15, zorder=5)
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nBest generated FOLD JSON:")
        print(json.dumps(best_fold, indent=2))

## 13. Plot Training Logs

In [ ]:
# Extract training logs from the trainer
logs = trainer.state.log_history

# Parse out loss and reward metrics
steps, losses, rewards = [], [], []
for entry in logs:
    if "loss" in entry:
        steps.append(entry.get("step", 0))
        losses.append(entry["loss"])
    if "reward" in entry:
        rewards.append(entry["reward"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if losses:
    ax1.plot(steps[:len(losses)], losses, color="steelblue", linewidth=1, alpha=0.7)
    # Smoothed line
    if len(losses) > 10:
        window = min(20, len(losses) // 5)
        smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
        ax1.plot(steps[window-1:len(smoothed)+window-1], smoothed, color="navy", linewidth=2)
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Loss")
    ax1.set_title("Training Loss")
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, "No loss data", ha="center", va="center", transform=ax1.transAxes)

if rewards:
    ax2.plot(range(len(rewards)), rewards, color="coral", linewidth=1, alpha=0.7)
    if len(rewards) > 10:
        window = min(20, len(rewards) // 5)
        smoothed = np.convolve(rewards, np.ones(window)/window, mode="valid")
        ax2.plot(range(window-1, len(smoothed)+window-1), smoothed, color="darkred", linewidth=2)
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Reward")
    ax2.set_title("Mean Reward")
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, "No reward data", ha="center", va="center", transform=ax2.transAxes)

plt.tight_layout()
plt.show()